In [18]:
import pandas as pd
import feedparser
from src.collectors.rss_collector import collect_rss_articles, clean_url
from src.collectors.ccnews_extractor import parse_extracted_article

import json
from urllib.parse import urlparse, urlsplit, urlunsplit, parse_qsl, urlencode
from pathlib import Path
import time
import os
from datetime import datetime
import random
import requests
import trafilatura
from tqdm import tqdm

In [8]:
ALLOWED_DOMAINS = {
    "reuters.com",
    "bbc.com",
    "bbc.co.uk",
    "ft.com",
    "bloomberg.com",
    "cnbc.com",
    "wsj.com",
    "nytimes.com",
    "economist.com",
    "theguardian.com",
    "washingtonpost.com",
    "apnews.com",
    "businessinsider.com",
    "marketwatch.com",
    "yahoo.com",
    "forbes.com",
    "cnn.com",
    "nbcnews.com",
    "abcnews.go.com",
    "aljazeera.com",

    # list of en domains
    'www.channelnewsasia.com',
    'www.thenationalnews.com',
    # 'www.swissinfo.ch',
    'www.straitstimes.com',
    'vietnamnews.vn',
    'abcnews.go.com',
    'www.cbsnews.com',
    'www.foxnews.com',
    'www.latimes.com',
    'globalnews.ca',
    'www.ctvnews.ca',
    'www.telegraph.co.uk',
    'www.independent.co.uk',
    'www.the-independent.com',
    'www.irishtimes.com',
    'www.scotsman.com',
    'www.yorkshirepost.co.uk',
    # 'timesofindia.indiatimes.com',
    # 'economictimes.indiatimes.com',
    'indianexpress.com',
    'www.ndtv.com',
    # 'www.business-standard.com',
    # 'www.livemint.com',
    # 'www.businesstoday.in',
    'gulfnews.com',
    'www.khaleejtimes.com',
    'allafrica.com',
    'www.timeslive.co.za',
    'www.businesslive.co.za',
    'thewest.com.au',
    'www.nzherald.co.nz',
    # 'www.nasdaq.com',
    # 'seekingalpha.com',
    'qz.com',
    # 'www.barchart.com',
    # 'www.pymnts.com',

    # ru domains
    'www.tass.ru',
    # 'ria.ru', # bad format of html
    'www.interfax.ru',
    'www.rbc.ru',
    'www.vedomosti.ru',
    'www.kommersant.ru',
    'www.lenta.ru',
    'www.nur.kz',
    'www.zakon.kz'
}

## check rss links

In [3]:

domains = sorted(ALLOWED_DOMAINS)

manual_feeds = {
    "bbc.com": ["https://feeds.bbci.co.uk/news/world/rss.xml"],
    "bbc.co.uk": ["https://feeds.bbci.co.uk/news/world/rss.xml"],
    "nytimes.com": ["https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml"],
    "cnn.com": ["https://rss.cnn.com/rss/edition.rss"],
    "aljazeera.com": ["https://www.aljazeera.com/xml/rss/all.xml"],
    "abcnews.go.com": ["https://abcnews.go.com/abcnews/topstories"],
}

paths = [
    "/feed",
    "/rss",
    "/rss.xml",
    "/feed.xml",
    "/atom.xml",
    "/index.xml",
    "/news/rss",
    "/rss/news",
]

def candidate_urls(domain: str) -> list[str]:
    base = [f"https://{domain}", f"http://{domain}"]
    if domain.startswith("www."):
        bare = domain[4:]
        base += [f"https://{bare}", f"http://{bare}"]
    urls = []
    for b in base:
        urls.extend([b + p for p in paths])
    urls.extend(manual_feeds.get(domain, []))
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            out.append(u)
            seen.add(u)
    return out


In [6]:

rows = []
working_feeds = []


print(f"Starting RSS discovery for {len(domains)} domains...\n")
for i, d in enumerate(domains, 1):
    found = None
    found_count = 0
    found_title = ""

    urls = candidate_urls(d)
    print(f"[{i}/{len(domains)}] domain={d} candidates={len(urls)}")

    for j, u in enumerate(urls, 1):
        f = feedparser.parse(u)
        bozo_ok = int(getattr(f, "bozo", 0)) == 0
        n = len(getattr(f, "entries", []))
        title = str(getattr(f, "feed", {}).get("title", "")).strip()

        print(
            f"   - try {j:02d}: entries={n:4d} bozo_ok={bozo_ok} "
            f"url={u}"
        )

        if n > 0:
            found = u
            found_count = n
            found_title = title
            break
        if bozo_ok and title:
            found = u
            found_count = n
            found_title = title
            break
    rows.append(
        {
            "domain": d,
            "rss_found": bool(found),
            "feed_url": found,
            "entries": found_count,
            "feed_title": found_title,
        }
    )
    if found:
        working_feeds.append(found)

rss_check_df = pd.DataFrame(rows).sort_values(["rss_found", "entries"], ascending=[False, False]).reset_index(drop=True)
print("domains_total:", len(domains))
print("domains_with_rss:", int(rss_check_df["rss_found"].sum()))
print("working_feeds:", len(working_feeds))

articles = collect_rss_articles(working_feeds)
articles_df = pd.DataFrame([a.__dict__ for a in articles]).drop_duplicates(subset=["url"]).reset_index(drop=True)

print("parsed_articles_unique_urls:", len(articles_df))
display(rss_check_df)
articles_df.head(20)

Starting RSS discovery for 53 domains...

[1/53] domain=abcnews.go.com candidates=17
   - try 01: entries=   0 bozo_ok=True url=https://abcnews.go.com/feed
   - try 02: entries=   0 bozo_ok=True url=https://abcnews.go.com/rss
   - try 03: entries=   0 bozo_ok=False url=https://abcnews.go.com/rss.xml
   - try 04: entries=   0 bozo_ok=True url=https://abcnews.go.com/feed.xml
   - try 05: entries=   0 bozo_ok=True url=https://abcnews.go.com/atom.xml
   - try 06: entries=   0 bozo_ok=True url=https://abcnews.go.com/index.xml
   - try 07: entries=   0 bozo_ok=True url=https://abcnews.go.com/news/rss
   - try 08: entries=   0 bozo_ok=True url=https://abcnews.go.com/rss/news
   - try 09: entries=   0 bozo_ok=True url=http://abcnews.go.com/feed
   - try 10: entries=   0 bozo_ok=True url=http://abcnews.go.com/rss
   - try 11: entries=   0 bozo_ok=True url=http://abcnews.go.com/rss.xml
   - try 12: entries=   0 bozo_ok=True url=http://abcnews.go.com/feed.xml
   - try 13: entries=   0 bozo_ok=Tru

RemoteDisconnected: Remote end closed connection without response

## parse rss

In [13]:
RSS_URLs = ["https://www.aljazeera.com/xml/rss/all.xml"]  # same feed as /rss self-link
MAX_ITEMS = 200
MIN_TEXT_LEN = 500
REQUEST_TIMEOUT = 30
ALLOWED_LANGS = {"en", "ru"}  # keep same policy as CC pipeline

root_dir = Path(os.path.abspath(os.path.join(os.path.dirname('__file__')))).parent
OUT_ROOT = os.path.join(root_dir, "data", "raw", "rss")

feed = feedparser.parse(RSS_URL)
entries = feed.entries[:MAX_ITEMS]


In [ ]:

items = []
stats = {
    "rss_entries": len(entries),
    "fetched_ok": 0,
    "extract_non_empty": 0,
    "json_errors": 0,
    "lang_filtered_out": 0,
    "written": 0,
}

for e in tqdm(entries, desc="RSS -> article extraction"):
    url = clean_url(str(e.get("link", "")).strip())
    if not url:
        continue

    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        html = r.text
        stats["fetched_ok"] += 1
    except Exception:
        continue

    extracted = trafilatura.extract(
        html,
        output_format="json",
        with_metadata=True
    )

    parsed, d = parse_extracted_article(
        extracted,
        min_text_len=MIN_TEXT_LEN,
        allowed_langs=ALLOWED_LANGS,
        lang_sample_chars=500,
        )
        
    for k, v in d.items():
        stats[k] = stats.get(k, 0) + v
    if not parsed:
        continue

    data, text, lang = parsed["data"], parsed["text"], parsed["lang"]

    item = {
        "url": url,
        "domain": urlparse(url).netloc.lower(),
        "title": data.get("title") or e.get("title"),
        "date": data.get("date") or e.get("published"),
        "author": data.get("author"),
        "lang": lang,
        "text": text,
    }
    items.append(item)
    stats["written"] += 1

print(stats)
print("sample:", items[0] if items else None)

In [14]:

def _month_partition_path(base_dir: str, item_date: str | None) -> str:
    dt = pd.to_datetime(item_date, errors="coerce")
    if pd.isna(dt):
        dt = pd.Timestamp.utcnow()
    year = f"{dt.year:04d}"
    month = f"{dt.month:02d}"
    out_dir = os.path.join(base_dir, year, month)
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, "articles.jsonl")

def run_rss_cycle(
    rss_urls,
    *,
    max_items=200,
    min_text_len=500,
    allowed_langs=("en", "ru"),
    request_timeout=30,
    out_root=OUT_ROOT
):
    items = []
    seen_urls = set()
    stats = {
        "feeds": len(rss_urls),
        "rss_entries": 0,
        "fetched_ok": 0,
        "extract_non_empty": 0,
        "json_errors": 0,
        "lang_errors": 0,
        "lang_filtered_out": 0,
        "written": 0,
    }

    for rss_url in rss_urls:
        feed = feedparser.parse(rss_url)
        entries = feed.entries[:max_items]
        stats["rss_entries"] += len(entries)

        for e in tqdm(entries, desc=f"RSS -> extract | {rss_url}"):
            
            time.sleep(random.uniform(0.5, 2.0))
            url = clean_url(str(e.get("link", "")).strip())
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)

            try:
                r = requests.get(
                    url,
                    timeout=request_timeout,
                    headers={"User-Agent": "Mozilla/5.0"},
                )
                r.raise_for_status()
                html = r.text
                stats["fetched_ok"] += 1
            except Exception:
                continue

            extracted = trafilatura.extract(
                html,
                output_format="json",
                with_metadata=True
            )

            parsed, d = parse_extracted_article(
                extracted,
                min_text_len=min_text_len,
                allowed_langs=allowed_langs,
                lang_sample_chars=500,
            )
            for k, v in d.items():
                stats[k] = stats.get(k, 0) + v
            if not parsed:
                continue

            data, text, lang = parsed["data"], parsed["text"], parsed["lang"]
            item = {
                "url": url,
                "domain": urlparse(url).netloc.lower(),
                "title": data.get("title") or e.get("title"),
                "date": data.get("date") or e.get("published"),
                "author": data.get("author"),
                "lang": lang,
                "text": text,
                "rss_url": rss_url,
                "collected_at": datetime.utcnow().isoformat(),
            }
            items.append(item)

    # save partitioned by item month
    for x in items:
        out_path = _month_partition_path(out_root, x.get("date"))
        with open(out_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
        stats["written"] += 1

    return stats, items

In [ ]:
while True:
    t0 = time.time()
    stats, _ = run_rss_cycle(RSS_URLs, out_root=OUT_ROOT)
    elapsed = time.time() - t0
    print(f"[RSS cycle done] elapsed_s={elapsed:.2f} stats={stats}")

RSS -> extract | https://www.aljazeera.com/xml/rss/all.xml: 100%|██████████| 25/25 [00:50<00:00,  2.03s/it]

[RSS cycle done] elapsed_s=51.72 stats={'feeds': 1, 'rss_entries': 25, 'fetched_ok': 25, 'extract_non_empty': 25, 'json_errors': 0, 'lang_errors': 0, 'lang_filtered_out': 0, 'written': 22}


NameError: name 'sleep_seconds' is not defined